In [1]:
pip install -U datasets pandas nltk scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing

In [2]:
import os
import re
import json
import random
from collections import Counter
from typing import List, Dict, Optional, Tuple

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import nltk

nltk.download("punkt", quiet=True)

# =========================================================
# Config
# =========================================================

DATASET_NAME = "arsalanaa/children_story_dataset"
OUTPUT_DIR = "processed_children_story_data"
RANDOM_SEED = 42

# Word count bounds for the story body
MIN_WORDS = 150
MAX_WORDS = 550

# Sentence bounds
MIN_SENTENCES = 5

# Prefer 2-3 characters, but we can keep 1-4 in v1 and filter later if needed
MIN_CHARACTERS = 1
MAX_CHARACTERS = 4

# Final split sizes
VAL_SIZE = 0.10
TEST_SIZE = 0.10

# Safety filter
BANNED_PATTERNS = [
    r"\bkill(ed|ing)?\b",
    r"\bdead\b",
    r"\bdie(d|s|ing)?\b",
    r"\bblood(y)?\b",
    r"\bweapon(s)?\b",
    r"\bgun(s)?\b",
    r"\bknife(s)?\b",
    r"\bmurder(ed|er)?\b",
    r"\bghost(s)?\b",
    r"\bdemon(s)?\b",
    r"\bmonster(s)?\b",
    r"\bhorror\b",
    r"\bromance\b",
    r"\bkiss(ed|ing)?\b",
    r"\bmarry|married|marriage\b",
    r"\bdrunk\b",
    r"\balcohol\b",
    r"\bbeer\b",
    r"\bwine\b",
    r"\bsmok(e|ing|ed)\b",
    r"\bcigarette(s)?\b",
]

# Useful common themes
THEME_KEYWORDS = [
    "kindness",
    "honesty",
    "sharing",
    "friendship",
    "teamwork",
    "patience",
    "responsibility",
    "bravery",
    "courage",
    "helping others",
    "helpfulness",
    "forgiveness",
    "curiosity",
    "perseverance",
    "not giving up",
    "respect",
    "listening",
    "cooperation",
    "gratitude",
    "empathy",
    "truth",
    "fairness",
]

# Stopwords for crude character extraction cleanup
CHARACTER_STOPWORDS = {
    "The", "A", "An", "One", "Once", "When", "Then", "After", "Before",
    "He", "She", "They", "It", "We", "I", "You",
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
    "January", "February", "March", "April", "May", "June", "July", "August",
    "September", "October", "November", "December",
    "Mom", "Dad", "Mother", "Father", "Grandma", "Grandpa",
}

NORMALIZED_INSTRUCTION = """Write a children's story for ages 6 to 8.

Requirements:
- Use simple vocabulary and clear sentences
- Keep the story under 500 words
- Use 2 to 3 main characters if possible
- Make the story suitable for 6 to 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, dark, or adult themes

Output format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

# =========================================================
# Utility functions
# =========================================================

def set_seed(seed: int = 42):
    random.seed(seed)

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" +\n", "\n", text)
    return text.strip()

def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))

def split_sentences(text: str) -> List[str]:
    try:
        return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]
    except Exception:
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def count_sentences(text: str) -> int:
    return len(split_sentences(text))

def contains_banned_content(text: str) -> bool:
    lower = text.lower()
    return any(re.search(pattern, lower) for pattern in BANNED_PATTERNS)

def extract_theme(prompt: str) -> str:
    if not prompt:
        return "general children's story"

    prompt_lower = prompt.lower()

    # direct keyword match
    for kw in THEME_KEYWORDS:
        if kw in prompt_lower:
            return kw

    # common patterns
    patterns = [
        r"about ([a-zA-Z \-]+?)(?:[.,;\n]|$)",
        r"theme:? ([a-zA-Z \-]+?)(?:[.,;\n]|$)",
        r"story on ([a-zA-Z \-]+?)(?:[.,;\n]|$)",
        r"story about ([a-zA-Z \-]+?)(?:[.,;\n]|$)",
    ]
    for pat in patterns:
        m = re.search(pat, prompt_lower)
        if m:
            candidate = m.group(1).strip()
            if 1 <= len(candidate.split()) <= 5:
                return candidate

    # fallback theme guess from educational cues
    if "science" in prompt_lower:
        return "curiosity"
    if "friend" in prompt_lower or "friendship" in prompt_lower:
        return "friendship"
    if "help" in prompt_lower:
        return "helping others"

    return "general children's story"

def extract_title_from_story(text: str) -> Optional[str]:
    if not text:
        return None

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return None

    first_line = lines[0]

    # If first line looks like a title
    if (
        len(first_line.split()) <= 12
        and not first_line.endswith(".")
        and not first_line.endswith("?")
        and not first_line.endswith("!")
    ):
        return first_line

    # If story begins with "Title:"
    m = re.search(r"^Title:\s*(.+)$", text, flags=re.IGNORECASE | re.MULTILINE)
    if m:
        return m.group(1).strip()

    return None

def generate_simple_title(text: str, theme: str) -> str:
    # Very crude fallback title generator based on likely lead character
    chars = extract_characters(text, max_chars=3)
    if chars:
        if len(chars) == 1:
            return f"{chars[0]} and the Lesson of {theme.title()}"
        elif len(chars) >= 2:
            return f"{chars[0]} and {chars[1]}"
    return f"A Story About {theme.title()}"

def remove_existing_title_from_story(text: str, title: Optional[str]) -> str:
    if not text or not title:
        return text
    lines = [ln for ln in text.splitlines()]
    if lines and lines[0].strip() == title.strip():
        body = "\n".join(lines[1:]).strip()
        return body if body else text
    return text

def extract_characters(text: str, max_chars: int = 3) -> List[str]:
    """
    Very rough character extraction:
    - capitalize words likely to be names
    - count repeated proper nouns
    - fallback to animal/role nouns if no names found
    """
    if not text:
        return []

    # Proper noun style candidate extraction
    candidates = re.findall(r"\b[A-Z][a-z]{2,}\b", text)
    counts = Counter()

    for c in candidates:
        if c not in CHARACTER_STOPWORDS:
            counts[c] += 1

    # Keep repeated candidates first
    named = [name for name, cnt in counts.most_common() if cnt >= 2]

    if len(named) >= 1:
        return named[:max_chars]

    # fallback animal/role nouns
    animal_role_patterns = [
        "rabbit", "fox", "bear", "bird", "owl", "turtle", "cat", "dog",
        "girl", "boy", "child", "squirrel", "mouse", "deer", "duck",
        "lion", "monkey", "elephant", "penguin", "frog",
    ]
    text_lower = text.lower()
    fallback = []
    for w in animal_role_patterns:
        if re.search(rf"\b{re.escape(w)}\b", text_lower):
            fallback.append(w.title())

    # dedupe, preserve order
    seen = set()
    deduped = []
    for x in fallback:
        if x not in seen:
            seen.add(x)
            deduped.append(x)

    return deduped[:max_chars]

def extract_moral(text: str, theme: str) -> str:
    """
    Try to extract a moral-like ending from the last 2-3 sentences.
    If none found, generate a short theme-based moral.
    """
    sentences = split_sentences(text)
    if not sentences:
        return default_moral_from_theme(theme)

    tail = " ".join(sentences[-3:])

    # explicit markers
    moral_patterns = [
        r"(moral(?: of the story)?[:\-]?\s*.+)$",
        r"(the lesson was .+)$",
        r"(they learned that .+)$",
        r"(from then on,? .+)$",
        r"(in the end,? .+)$",
    ]
    for pat in moral_patterns:
        m = re.search(pat, tail, flags=re.IGNORECASE)
        if m:
            moral = m.group(1).strip()
            moral = re.sub(r"^moral(?: of the story)?[:\-]?\s*", "", moral, flags=re.IGNORECASE)
            return cleanup_moral(moral)

    # If the last sentence looks lesson-like, use it
    last_sentence = sentences[-1].strip()
    if looks_like_moral(last_sentence):
        return cleanup_moral(last_sentence)

    return default_moral_from_theme(theme)

def looks_like_moral(sentence: str) -> bool:
    s = sentence.lower()
    clues = [
        "kindness", "honesty", "sharing", "friendship", "patience",
        "helping", "caring", "brave", "responsible", "important",
        "learned", "lesson", "better", "good", "always",
    ]
    return any(c in s for c in clues)

def cleanup_moral(moral: str) -> str:
    moral = moral.strip()
    moral = moral.replace("\n", " ")
    moral = re.sub(r"\s+", " ", moral)
    if not moral.endswith("."):
        moral += "."
    # keep it reasonably short
    words = moral.split()
    if len(words) > 25:
        moral = " ".join(words[:25]).rstrip(",;:") + "."
    return moral

def default_moral_from_theme(theme: str) -> str:
    theme = theme.lower().strip()

    theme_to_moral = {
        "kindness": "Kindness can make someone's day brighter.",
        "honesty": "Being honest helps people trust you.",
        "sharing": "Sharing can make everyone happier.",
        "friendship": "Good friends care for and help each other.",
        "teamwork": "Working together can solve hard problems.",
        "patience": "Good things often take time, so patience matters.",
        "responsibility": "Taking responsibility helps us grow and do what is right.",
        "bravery": "Small acts of bravery can help us face challenges.",
        "courage": "Courage helps us do the right thing even when we feel unsure.",
        "helping others": "Helping others can bring joy to everyone.",
        "helpfulness": "Helping others is a simple way to spread goodness.",
        "forgiveness": "Forgiving others can heal hurt feelings and bring peace.",
        "curiosity": "Curiosity helps us learn new things.",
        "perseverance": "If we keep trying, we can often succeed.",
        "not giving up": "Not giving up helps us grow stronger.",
        "respect": "Respect helps people feel valued and understood.",
        "listening": "Listening carefully helps us understand and care for others.",
        "cooperation": "Cooperation helps everyone do better together.",
        "gratitude": "Being thankful helps us notice the good around us.",
        "empathy": "Understanding others' feelings helps us be kinder.",
        "truth": "Telling the truth is important.",
        "fairness": "Being fair helps everyone feel included and respected.",
        "general children's story": "Doing the right thing can lead to good outcomes.",
    }
    return theme_to_moral.get(theme, "Doing the right thing can lead to good outcomes.")

def estimate_scene_friendliness(text: str) -> int:
    """
    Rough estimate of how many scene-like events appear in the story.
    """
    sentences = split_sentences(text)
    action_markers = 0
    for s in sentences:
        if re.search(
            r"\b(went|saw|found|met|asked|said|helped|looked|ran|walked|gave|shared|thought|decided|learned|returned|smiled|built|opened|carried|waited|grew)\b",
            s,
            flags=re.IGNORECASE,
        ):
            action_markers += 1
    return max(1, min(10, action_markers))

def normalize_story_body(text: str) -> str:
    text = clean_text(text)

    # remove explicit title markers if present
    text = re.sub(r"^Title:\s*.+?\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Story:\s*", "", text, flags=re.IGNORECASE)

    # remove leading/trailing quotes
    text = text.strip().strip('"').strip("'").strip()

    # normalize repeated blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text

def build_output(title: str, characters: List[str], story: str, moral: str) -> str:
    characters_text = ", ".join(characters) if characters else "Unknown"
    return (
        f"Title: {title}\n"
        f"Characters: {characters_text}\n"
        f"Story:\n{story}\n\n"
        f"Moral: {moral}"
    )

def make_example(theme: str, title: str, characters: List[str], story: str, moral: str) -> Dict:
    return {
        "instruction": NORMALIZED_INSTRUCTION,
        "input": f"Theme: {theme}",
        "output": build_output(title, characters, story, moral),
    }

# =========================================================
# Filtering and preprocessing
# =========================================================

def should_reject_row(prompt: str, text: str, text_token_length: Optional[int]) -> Tuple[bool, str]:
    if not prompt or not str(prompt).strip():
        return True, "missing_prompt"
    if not text or not str(text).strip():
        return True, "missing_text"

    cleaned = normalize_story_body(text)
    wc = count_words(cleaned)
    sc = count_sentences(cleaned)

    if wc < MIN_WORDS:
        return True, "too_short"
    if wc > MAX_WORDS:
        return True, "too_long"
    if sc < MIN_SENTENCES:
        return True, "too_few_sentences"
    if contains_banned_content(cleaned):
        return True, "unsafe_content"

    # reject highly list-like / instruction-like outputs
    lines = [ln.strip() for ln in cleaned.splitlines() if ln.strip()]
    bullet_like = sum(1 for ln in lines if re.match(r"^[-*•\d]+\s", ln))
    if bullet_like >= 3:
        return True, "not_story_like"

    return False, "accepted"

def preprocess_row(row: Dict) -> Optional[Dict]:
    prompt = clean_text(str(row.get("prompt", "")))
    text = clean_text(str(row.get("text", "")))
    text_token_length = row.get("text_token_length", None)

    reject, reason = should_reject_row(prompt, text, text_token_length)
    if reject:
        return {"reject_reason": reason, "prompt": prompt, "text": text}

    theme = extract_theme(prompt)

    story_body = normalize_story_body(text)

    title = extract_title_from_story(story_body)
    if title is None:
        title = generate_simple_title(story_body, theme)

    story_body = remove_existing_title_from_story(story_body, title)
    story_body = normalize_story_body(story_body)

    characters = extract_characters(story_body, max_chars=3)

    # Optional extra filter on character count
    if len(characters) < MIN_CHARACTERS or len(characters) > MAX_CHARACTERS:
        return {"reject_reason": "bad_character_count", "prompt": prompt, "text": text}

    moral = extract_moral(story_body, theme)

    # scene-friendliness can be stored as metadata for inspection
    estimated_scenes = estimate_scene_friendliness(story_body)

    # final example
    example = make_example(
        theme=theme,
        title=title,
        characters=characters,
        story=story_body,
        moral=moral,
    )

    # add metadata for debugging and analysis
    example["meta"] = {
        "source_prompt": prompt,
        "word_count": count_words(story_body),
        "sentence_count": count_sentences(story_body),
        "character_count": len(characters),
        "estimated_scene_count": estimated_scenes,
        "text_token_length": text_token_length,
        "theme": theme,
    }
    return example

# =========================================================
# Save helpers
# =========================================================

def save_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def save_csv(path: str, rows: List[Dict]):
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)

# =========================================================
# Main pipeline
# =========================================================

def main():
    set_seed(RANDOM_SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Loading dataset: {DATASET_NAME}")
    ds = load_dataset(DATASET_NAME, split="train")

    accepted = []
    rejected = []

    print(f"Processing {len(ds)} rows...")

    for i, row in enumerate(ds):
        processed = preprocess_row(row)

        if processed is None:
            rejected.append({"reject_reason": "unknown_none"})
            continue

        if "reject_reason" in processed:
            rejected.append(processed)
        else:
            accepted.append(processed)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i + 1}/{len(ds)} | accepted={len(accepted)} | rejected={len(rejected)}")

    print("\nDone preprocessing.")
    print(f"Accepted rows: {len(accepted)}")
    print(f"Rejected rows: {len(rejected)}")

    # Save rejected rows for inspection
    save_csv(os.path.join(OUTPUT_DIR, "rejected_rows.csv"), rejected)

    # Save all accepted rows before splitting
    save_jsonl(os.path.join(OUTPUT_DIR, "all_processed.jsonl"), accepted)

    # For splitting, keep only instruction/input/output in the final train files
    final_rows = [
        {
            "instruction": x["instruction"],
            "input": x["input"],
            "output": x["output"],
        }
        for x in accepted
    ]

    # Optional metadata save
    metadata_rows = [x["meta"] for x in accepted]
    save_csv(os.path.join(OUTPUT_DIR, "accepted_metadata.csv"), metadata_rows)

    # Split train / val / test
    train_rows, temp_rows = train_test_split(
        final_rows,
        test_size=(VAL_SIZE + TEST_SIZE),
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
    val_rows, test_rows = train_test_split(
        temp_rows,
        test_size=relative_test_size,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    print("\nSplit sizes:")
    print(f"Train: {len(train_rows)}")
    print(f"Val:   {len(val_rows)}")
    print(f"Test:  {len(test_rows)}")

    save_jsonl(os.path.join(OUTPUT_DIR, "train.jsonl"), train_rows)
    save_jsonl(os.path.join(OUTPUT_DIR, "val.jsonl"), val_rows)
    save_jsonl(os.path.join(OUTPUT_DIR, "test.jsonl"), test_rows)

    # Save a few samples for manual inspection
    sample_path = os.path.join(OUTPUT_DIR, "sample_examples.jsonl")
    save_jsonl(sample_path, accepted[:25])

    print(f"\nSaved files in: {OUTPUT_DIR}")
    print("- all_processed.jsonl")
    print("- accepted_metadata.csv")
    print("- rejected_rows.csv")
    print("- train.jsonl")
    print("- val.jsonl")
    print("- test.jsonl")
    print("- sample_examples.jsonl")

if __name__ == "__main__":
    main()

Loading dataset: arsalanaa/children_story_dataset


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Children-Stories-2-Final.json:   0%|          | 0.00/343M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/89384 [00:00<?, ? examples/s]

Processing 89384 rows...
Processed 5000/89384 | accepted=4553 | rejected=447
Processed 10000/89384 | accepted=9118 | rejected=882
Processed 15000/89384 | accepted=13654 | rejected=1346
Processed 20000/89384 | accepted=18198 | rejected=1802
Processed 25000/89384 | accepted=22730 | rejected=2270
Processed 30000/89384 | accepted=27285 | rejected=2715
Processed 35000/89384 | accepted=31838 | rejected=3162
Processed 40000/89384 | accepted=36348 | rejected=3652
Processed 45000/89384 | accepted=40937 | rejected=4063
Processed 50000/89384 | accepted=45484 | rejected=4516
Processed 55000/89384 | accepted=50055 | rejected=4945
Processed 60000/89384 | accepted=54580 | rejected=5420
Processed 65000/89384 | accepted=59130 | rejected=5870
Processed 70000/89384 | accepted=63658 | rejected=6342
Processed 75000/89384 | accepted=68178 | rejected=6822
Processed 80000/89384 | accepted=72714 | rejected=7286
Processed 85000/89384 | accepted=77280 | rejected=7720

Done preprocessing.
Accepted rows: 81258
Rej

In [3]:
import json
import random
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("processed_children_story_data")
random.seed(42)

# -------------------------------------------------
# Helpers
# -------------------------------------------------

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def print_divider(title=""):
    print("\n" + "=" * 80)
    if title:
        print(title)
        print("=" * 80)

# -------------------------------------------------
# Load files
# -------------------------------------------------

sample_examples = load_jsonl(OUTPUT_DIR / "sample_examples.jsonl")
accepted_meta = pd.read_csv(OUTPUT_DIR / "accepted_metadata.csv")
rejected_rows = pd.read_csv(OUTPUT_DIR / "rejected_rows.csv")

# -------------------------------------------------
# Basic stats
# -------------------------------------------------

print_divider("BASIC COUNTS")
print(f"Sample examples: {len(sample_examples)}")
print(f"Accepted metadata rows: {len(accepted_meta)}")
print(f"Rejected rows: {len(rejected_rows)}")

print_divider("ACCEPTED METADATA SUMMARY")
print(accepted_meta.describe(include="all").transpose())

print_divider("REJECT REASONS")
if "reject_reason" in rejected_rows.columns:
    print(rejected_rows["reject_reason"].value_counts(dropna=False).head(20))
else:
    print("No reject_reason column found.")

# -------------------------------------------------
# Random accepted examples
# -------------------------------------------------

def show_accepted_examples(n=5):
    print_divider(f"RANDOM ACCEPTED EXAMPLES ({n})")
    chosen = random.sample(sample_examples, min(n, len(sample_examples)))
    for i, ex in enumerate(chosen, 1):
        print(f"\n--- Accepted Example {i} ---")
        print(f"Instruction:\n{ex.get('instruction', '')}\n")
        print(f"Input:\n{ex.get('input', '')}\n")
        print(f"Output:\n{ex.get('output', '')[:2500]}\n")
        if "meta" in ex:
            print("Meta:")
            for k, v in ex["meta"].items():
                print(f"  {k}: {v}")

show_accepted_examples(5)

# -------------------------------------------------
# Random rejected examples
# -------------------------------------------------

def show_rejected_examples(n=5):
    print_divider(f"RANDOM REJECTED EXAMPLES ({n})")
    if len(rejected_rows) == 0:
        print("No rejected rows found.")
        return

    sampled = rejected_rows.sample(min(n, len(rejected_rows)), random_state=42)
    for i, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f"\n--- Rejected Example {i} ---")
        print(f"Reason: {row.get('reject_reason', 'N/A')}")
        prompt = str(row.get("prompt", ""))[:800]
        text = str(row.get("text", ""))[:1200]
        print(f"Prompt:\n{prompt}\n")
        print(f"Text:\n{text}\n")

show_rejected_examples(5)

# -------------------------------------------------
# Suspicious accepted rows
# -------------------------------------------------

print_divider("SUSPICIOUS ACCEPTED ROWS")

suspicious = accepted_meta[
    (accepted_meta["word_count"] < 180) |
    (accepted_meta["word_count"] > 520) |
    (accepted_meta["character_count"] > 3) |
    (accepted_meta["estimated_scene_count"] < 5) |
    (accepted_meta["estimated_scene_count"] > 8)
]

print(f"Suspicious accepted rows count: {len(suspicious)}")
print(suspicious.head(20))

# -------------------------------------------------
# Theme distribution
# -------------------------------------------------

print_divider("TOP THEMES")
if "theme" in accepted_meta.columns:
    print(accepted_meta["theme"].value_counts().head(20))
else:
    print("No theme column found.")

# -------------------------------------------------
# Optional: save suspicious rows
# -------------------------------------------------

suspicious.to_csv(OUTPUT_DIR / "suspicious_accepted_rows.csv", index=False)
print("\nSaved suspicious rows to suspicious_accepted_rows.csv")


BASIC COUNTS
Sample examples: 25
Accepted metadata rows: 81258
Rejected rows: 8126

ACCEPTED METADATA SUMMARY
                         count unique  \
source_prompt            81258  81148   
word_count             81258.0    NaN   
sentence_count         81258.0    NaN   
character_count        81258.0    NaN   
estimated_scene_count  81258.0    NaN   
text_token_length      81258.0    NaN   
theme                    81258   3383   

                                                                     top  \
source_prompt          Write an educational story (3-5 paragraphs) ta...   
word_count                                                           NaN   
sentence_count                                                       NaN   
character_count                                                      NaN   
estimated_scene_count                                                NaN   
text_token_length                                                    NaN   
theme                       

In [4]:
pip install -U datasets pandas nltk scikit-learn

In [5]:
import os
import re
import json
import random
from collections import Counter, defaultdict
from typing import List, Dict, Optional, Tuple

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import nltk

nltk.download("punkt", quiet=True)

# =========================================================
# Config
# =========================================================

DATASET_NAME = "arsalanaa/children_story_dataset"
OUTPUT_DIR = "processed_children_story_data_v2"
RANDOM_SEED = 42

MIN_WORDS = 180
MAX_WORDS = 520
MIN_SENTENCES = 6

# stricter than v1
MIN_CHARACTERS = 2
MAX_CHARACTERS = 3

MIN_SCENES = 5
MAX_SCENES = 8

VAL_SIZE = 0.10
TEST_SIZE = 0.10

# Cap dominant broad themes after filtering
MAX_ROWS_PER_THEME = {
    "general": 12000,
    "curiosity": 8000,
}

APPROVED_THEMES = {
    "kindness",
    "honesty",
    "sharing",
    "friendship",
    "teamwork",
    "patience",
    "responsibility",
    "courage",
    "curiosity",
    "respect",
    "empathy",
    "gratitude",
    "fairness",
    "listening",
    "perseverance",
    "general",
}

# safety / content mismatch
BANNED_CONTENT_PATTERNS = [
    r"\bkill(ed|ing)?\b",
    r"\bdead\b",
    r"\bdie(d|s|ing)?\b",
    r"\bblood(y)?\b",
    r"\bweapon(s)?\b",
    r"\bgun(s)?\b",
    r"\bknife(s)?\b",
    r"\bmurder(ed|er)?\b",
    r"\bghost(s)?\b",
    r"\bdemon(s)?\b",
    r"\bmonster(s)?\b",
    r"\bhorror\b",
    r"\bromance\b",
    r"\bkiss(ed|ing)?\b",
    r"\bmarry|married|marriage\b",
    r"\balcohol\b",
    r"\bbeer\b",
    r"\bwine\b",
    r"\bcigarette(s)?\b",
    r"\bworld war\b",
    r"\bbattle\b",
    r"\bcampaign\b",
    r"\bmilitary\b",
    r"\bnavy\b",
    r"\bair strike\b",
]

# reject prompts that are too adult / corporate / technical / newsy
DOMAIN_MISMATCH_PATTERNS = [
    r"\bpink floyd\b",
    r"\broger waters\b",
    r"\bdell\b",
    r"\blaptop\b",
    r"\bcapacitor\b",
    r"\bformula one\b",
    r"\bferrari\b",
    r"\brome\b",
    r"\btravel sites\b",
    r"\bcatacombs\b",
    r"\bbank industry architecture network\b",
    r"\bbian\b",
    r"\benterprise architecture\b",
    r"\bvendor lock-in\b",
    r"\blegacy systems\b",
    r"\bdata silos\b",
    r"\btrademark\b",
    r"\blawsuit\b",
    r"\bfinancial dispute\b",
    r"\bcustomer complaints?\b",
    r"\brestaurant\b",
    r"\bbrisket\b",
    r"\bribs\b",
    r"\bcoffee\b",
    r"\bwwii\b",
    r"\bworld war\b",
    r"\bmidway\b",
    r"\bjapan\b",
    r"\bmalaysian grand prix\b",
    r"\bmclaren\b",
    r"\bcnn\b",
    r"\breview\b",
    r"\bpolitics?\b",
    r"\bcompany\b",
    r"\bbusiness\b",
    r"\bmarket\b",
    r"\baccounting\b",
    r"\bmayor'?s laptop\b",
    r"\bformula\b",
]

THEME_SYNONYMS = {
    "kindness": ["kindness", "kind", "helping others", "helpfulness", "being kind"],
    "honesty": ["honesty", "honest", "truth", "telling the truth"],
    "sharing": ["sharing", "share"],
    "friendship": ["friendship", "friend", "friends"],
    "teamwork": ["teamwork", "cooperation", "working together"],
    "patience": ["patience", "waiting", "wait"],
    "responsibility": ["responsibility", "responsible", "taking responsibility"],
    "courage": ["courage", "bravery", "brave"],
    "curiosity": ["curiosity", "curious", "science", "learning", "discovery"],
    "respect": ["respect", "respectful"],
    "empathy": ["empathy", "understanding feelings", "compassion"],
    "gratitude": ["gratitude", "thankful", "thankfulness", "being thankful"],
    "fairness": ["fairness", "fair", "justice"],
    "listening": ["listening", "listen"],
    "perseverance": ["perseverance", "not giving up", "keep trying", "determination"],
}

CHARACTER_STOPWORDS = {
    "The", "A", "An", "One", "Once", "When", "Then", "After", "Before",
    "He", "She", "They", "It", "We", "I", "You", "This", "That", "These", "Those",
    "But", "And", "Or", "If", "Because", "While", "As", "So",
    "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday",
    "January", "February", "March", "April", "May", "June", "July", "August",
    "September", "October", "November", "December",
    "Mom", "Dad", "Mother", "Father", "Grandma", "Grandpa",
    "Mr", "Mrs", "Ms", "Dr", "Sir",
    "Rome", "Dell", "Pink", "Floyd", "San", "This", "But",
}

BAD_CHARACTER_TOKENS = {
    "This", "That", "But", "And", "Then", "Rome", "Dell", "San", "It",
    "Science", "Life", "Story", "One", "Two", "Three",
}

ANIMAL_ROLE_FALLBACKS = [
    "rabbit", "fox", "bear", "bird", "owl", "turtle", "cat", "dog",
    "girl", "boy", "child", "squirrel", "mouse", "deer", "duck",
    "lion", "monkey", "elephant", "penguin", "frog", "hedgehog",
    "puppy", "kitten", "teacher", "farmer", "baker", "friend",
]

NORMALIZED_INSTRUCTION = """Write a children's story for ages 6 to 8.

Requirements:
- Use simple vocabulary and clear sentences
- Keep the story under 500 words
- Use 2 to 3 main characters
- Make the story suitable for 6 to 7 visual scenes
- Include a clear beginning, middle, and end
- End with an explicit moral lesson
- Avoid scary, violent, dark, or adult themes

Output format:
Title: ...
Characters: ...
Story:
...
Moral: ...
"""

# =========================================================
# Helpers
# =========================================================

def set_seed(seed=42):
    random.seed(seed)

def clean_text(text: str) -> str:
    if text is None:
        return ""
    text = text.replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" +\n", "\n", text)
    return text.strip()

def count_words(text: str) -> int:
    return len(re.findall(r"\b\w+\b", text))

def split_sentences(text: str) -> List[str]:
    try:
        return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]
    except Exception:
        return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def count_sentences(text: str) -> int:
    return len(split_sentences(text))

def contains_any_pattern(text: str, patterns: List[str]) -> bool:
    t = text.lower()
    return any(re.search(p, t) for p in patterns)

def normalize_story_body(text: str) -> str:
    text = clean_text(text)
    text = re.sub(r"^Title:\s*.+?\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^Story:\s*", "", text, flags=re.IGNORECASE)
    text = text.strip().strip('"').strip("'").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text

def extract_title_from_story(text: str) -> Optional[str]:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    if not lines:
        return None
    first_line = lines[0]
    if (
        len(first_line.split()) <= 12
        and not first_line.endswith(".")
        and not first_line.endswith("?")
        and not first_line.endswith("!")
    ):
        return first_line
    m = re.search(r"^Title:\s*(.+)$", text, flags=re.IGNORECASE | re.MULTILINE)
    return m.group(1).strip() if m else None

def remove_existing_title_from_story(text: str, title: Optional[str]) -> str:
    if not text or not title:
        return text
    lines = text.splitlines()
    if lines and lines[0].strip() == title.strip():
        body = "\n".join(lines[1:]).strip()
        return body if body else text
    return text

def canonicalize_theme(prompt: str) -> str:
    p = prompt.lower()
    for canonical, variants in THEME_SYNONYMS.items():
        for v in variants:
            if v in p:
                return canonical
    return "general"

def looks_domain_mismatched(prompt: str, text: str) -> bool:
    combined = f"{prompt}\n{text}".lower()
    return contains_any_pattern(combined, DOMAIN_MISMATCH_PATTERNS)

def estimate_scene_friendliness(text: str) -> int:
    sentences = split_sentences(text)
    action_markers = 0
    for s in sentences:
        if re.search(
            r"\b(went|saw|found|met|asked|said|helped|looked|ran|walked|gave|shared|thought|decided|learned|returned|smiled|built|opened|carried|waited|grew|started|finished|followed|brought|made)\b",
            s,
            flags=re.IGNORECASE,
        ):
            action_markers += 1
    return max(1, min(10, action_markers))

def extract_characters(text: str, max_chars: int = 3) -> List[str]:
    if not text:
        return []

    candidates = re.findall(r"\b[A-Z][a-z]{2,}\b", text)
    counts = Counter()

    for c in candidates:
        if c not in CHARACTER_STOPWORDS:
            counts[c] += 1

    named = []
    for name, cnt in counts.most_common():
        if cnt >= 2 and name not in BAD_CHARACTER_TOKENS:
            named.append(name)

    if len(named) >= 2:
        return named[:max_chars]

    # fallback
    text_lower = text.lower()
    fallback = []
    for w in ANIMAL_ROLE_FALLBACKS:
        if re.search(rf"\b{re.escape(w)}\b", text_lower):
            fallback.append(w.title())

    deduped = []
    seen = set()
    for x in fallback:
        if x not in seen and x not in BAD_CHARACTER_TOKENS:
            seen.add(x)
            deduped.append(x)

    return deduped[:max_chars]

def bad_character_list(chars: List[str]) -> bool:
    if len(chars) < MIN_CHARACTERS or len(chars) > MAX_CHARACTERS:
        return True
    for c in chars:
        if c in BAD_CHARACTER_TOKENS:
            return True
        if len(c) <= 2:
            return True
    return False

def default_moral_from_theme(theme: str) -> str:
    mapping = {
        "kindness": "Kindness can brighten someone's day.",
        "honesty": "Being honest helps people trust you.",
        "sharing": "Sharing can make everyone happier.",
        "friendship": "Good friends help and care for each other.",
        "teamwork": "Working together helps solve hard problems.",
        "patience": "Good things often take time, so patience matters.",
        "responsibility": "Taking responsibility helps us do what is right.",
        "courage": "Courage helps us face challenges and do the right thing.",
        "curiosity": "Curiosity helps us learn and grow.",
        "respect": "Respect helps everyone feel valued.",
        "empathy": "Understanding others' feelings helps us be kind.",
        "gratitude": "Being thankful helps us notice the good around us.",
        "fairness": "Being fair helps everyone feel included.",
        "listening": "Listening carefully helps us understand others better.",
        "perseverance": "If we keep trying, we can improve and succeed.",
        "general": "Doing the right thing can lead to good outcomes.",
    }
    return mapping.get(theme, "Doing the right thing can lead to good outcomes.")

def cleanup_moral(moral: str) -> str:
    moral = moral.strip().replace("\n", " ")
    moral = re.sub(r"\s+", " ", moral)
    moral = moral.strip(" -:;,.")
    if not moral.endswith("."):
        moral += "."
    words = moral.split()
    if len(words) > 22:
        moral = " ".join(words[:22]).rstrip(",;:") + "."
    return moral

def looks_like_good_moral(sentence: str) -> bool:
    s = sentence.lower()
    clues = [
        "kindness", "honesty", "sharing", "friendship", "patience",
        "responsibility", "respect", "help", "learned", "lesson",
        "important", "trust", "care", "fair", "together", "brave",
        "thankful", "listen", "keep trying", "understand"
    ]
    return any(c in s for c in clues)

def extract_moral(text: str, theme: str) -> str:
    sentences = split_sentences(text)
    if not sentences:
        return default_moral_from_theme(theme)

    tail = sentences[-3:]

    # explicit patterns
    for s in reversed(tail):
        if looks_like_good_moral(s):
            return cleanup_moral(s)

    # fallback
    return default_moral_from_theme(theme)

def moral_is_bad(moral: str) -> bool:
    if not moral:
        return True
    words = moral.split()
    if len(words) < 5:
        return True
    bad_endings = ["after all.", "and then.", "but.", "this."]
    if moral.lower().strip() in bad_endings:
        return True
    if moral.count("...") > 0:
        return True
    return False

def generate_title(text: str, theme: str, characters: List[str]) -> str:
    if len(characters) >= 2:
        return f"{characters[0]} and {characters[1]}"
    if len(characters) == 1:
        return f"{characters[0]} Learns About {theme.title()}"
    return f"A Story About {theme.title()}"

def build_output(title: str, characters: List[str], story: str, moral: str) -> str:
    return (
        f"Title: {title}\n"
        f"Characters: {', '.join(characters)}\n"
        f"Story:\n{story}\n\n"
        f"Moral: {moral}"
    )

# =========================================================
# Core filtering
# =========================================================

def should_reject_row(prompt: str, text: str) -> Tuple[bool, str]:
    if not prompt.strip():
        return True, "missing_prompt"
    if not text.strip():
        return True, "missing_text"

    story = normalize_story_body(text)
    wc = count_words(story)
    sc = count_sentences(story)

    if wc < MIN_WORDS:
        return True, "too_short"
    if wc > MAX_WORDS:
        return True, "too_long"
    if sc < MIN_SENTENCES:
        return True, "too_few_sentences"
    if contains_any_pattern(story, BANNED_CONTENT_PATTERNS):
        return True, "unsafe_content"
    if looks_domain_mismatched(prompt, story):
        return True, "domain_mismatch"

    lines = [ln.strip() for ln in story.splitlines() if ln.strip()]
    bullet_like = sum(1 for ln in lines if re.match(r"^[-*•\d]+\s", ln))
    if bullet_like >= 3:
        return True, "not_story_like"

    return False, "accepted"

def preprocess_row(row: Dict) -> Dict:
    prompt = clean_text(str(row.get("prompt", "")))
    text = clean_text(str(row.get("text", "")))
    token_len = row.get("text_token_length", None)

    reject, reason = should_reject_row(prompt, text)
    if reject:
        return {"reject_reason": reason, "prompt": prompt, "text": text}

    theme = canonicalize_theme(prompt)
    story = normalize_story_body(text)

    scene_count = estimate_scene_friendliness(story)
    if scene_count < MIN_SCENES or scene_count > MAX_SCENES:
        return {"reject_reason": "bad_scene_count", "prompt": prompt, "text": text}

    chars = extract_characters(story, max_chars=3)
    if bad_character_list(chars):
        return {"reject_reason": "bad_character_count_or_names", "prompt": prompt, "text": text}

    title = extract_title_from_story(story)
    if title is None or len(title.split()) > 12:
        title = generate_title(story, theme, chars)

    story = remove_existing_title_from_story(story, title)
    story = normalize_story_body(story)

    moral = extract_moral(story, theme)
    if moral_is_bad(moral):
        return {"reject_reason": "bad_moral", "prompt": prompt, "text": text}

    example = {
        "instruction": NORMALIZED_INSTRUCTION,
        "input": f"Theme: {theme}",
        "output": build_output(title, chars, story, moral),
        "meta": {
            "source_prompt": prompt,
            "word_count": count_words(story),
            "sentence_count": count_sentences(story),
            "character_count": len(chars),
            "estimated_scene_count": scene_count,
            "text_token_length": token_len,
            "theme": theme,
            "title": title,
            "characters": chars,
            "moral": moral,
        },
    }
    return example

# =========================================================
# Save helpers
# =========================================================

def save_jsonl(path: str, rows: List[Dict]):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def save_csv(path: str, rows: List[Dict]):
    pd.DataFrame(rows).to_csv(path, index=False)

# =========================================================
# Main
# =========================================================

def main():
    set_seed(RANDOM_SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Loading dataset: {DATASET_NAME}")
    ds = load_dataset(DATASET_NAME, split="train")

    accepted = []
    rejected = []

    print(f"Processing {len(ds)} rows...")

    for i, row in enumerate(ds):
        out = preprocess_row(row)
        if "reject_reason" in out:
            rejected.append(out)
        else:
            accepted.append(out)

        if (i + 1) % 5000 == 0:
            print(f"Processed {i+1}/{len(ds)} | accepted={len(accepted)} | rejected={len(rejected)}")

    print("\nBefore balancing:")
    print(f"Accepted rows: {len(accepted)}")
    print(f"Rejected rows: {len(rejected)}")

    # balance broad themes
    by_theme = defaultdict(list)
    for ex in accepted:
        by_theme[ex["meta"]["theme"]].append(ex)

    balanced = []
    for theme, rows in by_theme.items():
        random.shuffle(rows)
        cap = MAX_ROWS_PER_THEME.get(theme, None)
        if cap is not None:
            rows = rows[:cap]
        balanced.extend(rows)

    random.shuffle(balanced)

    print("\nAfter balancing:")
    print(f"Balanced accepted rows: {len(balanced)}")

    # save diagnostics
    save_csv(os.path.join(OUTPUT_DIR, "rejected_rows.csv"), rejected)
    save_jsonl(os.path.join(OUTPUT_DIR, "all_processed.jsonl"), balanced)
    save_jsonl(os.path.join(OUTPUT_DIR, "sample_examples.jsonl"), balanced[:25])

    meta_rows = [x["meta"] for x in balanced]
    save_csv(os.path.join(OUTPUT_DIR, "accepted_metadata.csv"), meta_rows)

    final_rows = [
        {
            "instruction": x["instruction"],
            "input": x["input"],
            "output": x["output"],
        }
        for x in balanced
    ]

    train_rows, temp_rows = train_test_split(
        final_rows,
        test_size=(VAL_SIZE + TEST_SIZE),
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    relative_test_size = TEST_SIZE / (VAL_SIZE + TEST_SIZE)
    val_rows, test_rows = train_test_split(
        temp_rows,
        test_size=relative_test_size,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    save_jsonl(os.path.join(OUTPUT_DIR, "train.jsonl"), train_rows)
    save_jsonl(os.path.join(OUTPUT_DIR, "val.jsonl"), val_rows)
    save_jsonl(os.path.join(OUTPUT_DIR, "test.jsonl"), test_rows)

    print("\nSplit sizes:")
    print(f"Train: {len(train_rows)}")
    print(f"Val:   {len(val_rows)}")
    print(f"Test:  {len(test_rows)}")

    # simple reject summary
    reject_df = pd.DataFrame(rejected)
    if "reject_reason" in reject_df.columns:
        print("\nTop reject reasons:")
        print(reject_df["reject_reason"].value_counts().head(20))

    meta_df = pd.DataFrame(meta_rows)
    if len(meta_df) > 0:
        print("\nTheme distribution:")
        print(meta_df["theme"].value_counts().head(20))

    print(f"\nSaved files in: {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Loading dataset: arsalanaa/children_story_dataset
Processing 89384 rows...
Processed 5000/89384 | accepted=2103 | rejected=2897
Processed 10000/89384 | accepted=4266 | rejected=5734
Processed 15000/89384 | accepted=6381 | rejected=8619
Processed 20000/89384 | accepted=8472 | rejected=11528
Processed 25000/89384 | accepted=10666 | rejected=14334
Processed 30000/89384 | accepted=12777 | rejected=17223
Processed 35000/89384 | accepted=14878 | rejected=20122
Processed 40000/89384 | accepted=16979 | rejected=23021
Processed 45000/89384 | accepted=19144 | rejected=25856
Processed 50000/89384 | accepted=21308 | rejected=28692
Processed 55000/89384 | accepted=23401 | rejected=31599
Processed 60000/89384 | accepted=25440 | rejected=34560
Processed 65000/89384 | accepted=27607 | rejected=37393
Processed 70000/89384 | accepted=29705 | rejected=40295
Processed 75000/89384 | accepted=31785 | rejected=43215
Processed 80000/89384 | accepted=33869 | rejected=46131
Processed 85000/89384 | accepted=3597

In [6]:
import json
import random
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("processed_children_story_data_v2")
random.seed(42)

# -------------------------------------------------
# Helpers
# -------------------------------------------------

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def print_divider(title=""):
    print("\n" + "=" * 80)
    if title:
        print(title)
        print("=" * 80)

# -------------------------------------------------
# Load files
# -------------------------------------------------

sample_examples = load_jsonl(OUTPUT_DIR / "sample_examples.jsonl")
accepted_meta = pd.read_csv(OUTPUT_DIR / "accepted_metadata.csv")
rejected_rows = pd.read_csv(OUTPUT_DIR / "rejected_rows.csv")

# -------------------------------------------------
# Basic counts
# -------------------------------------------------

print_divider("BASIC COUNTS")
print(f"Sample examples: {len(sample_examples)}")
print(f"Accepted metadata rows: {len(accepted_meta)}")
print(f"Rejected rows: {len(rejected_rows)}")

# -------------------------------------------------
# Metadata summary
# -------------------------------------------------

print_divider("ACCEPTED METADATA SUMMARY")
print(accepted_meta.describe(include="all").transpose())

# -------------------------------------------------
# Reject reasons
# -------------------------------------------------

print_divider("REJECT REASONS")
if "reject_reason" in rejected_rows.columns:
    print(rejected_rows["reject_reason"].value_counts(dropna=False).head(30))
else:
    print("No reject_reason column found.")

# -------------------------------------------------
# Theme distribution
# -------------------------------------------------

print_divider("TOP THEMES")
if "theme" in accepted_meta.columns:
    print(accepted_meta["theme"].value_counts().head(30))
else:
    print("No theme column found.")

# -------------------------------------------------
# Scene distribution
# -------------------------------------------------

print_divider("SCENE COUNT DISTRIBUTION")
if "estimated_scene_count" in accepted_meta.columns:
    print(accepted_meta["estimated_scene_count"].value_counts().sort_index())
else:
    print("No estimated_scene_count column found.")

# -------------------------------------------------
# Moral length sanity
# -------------------------------------------------

if "moral" in accepted_meta.columns:
    accepted_meta["moral_word_count"] = accepted_meta["moral"].fillna("").apply(lambda x: len(str(x).split()))
    print_divider("MORAL WORD COUNT SUMMARY")
    print(accepted_meta["moral_word_count"].describe())

# -------------------------------------------------
# Random accepted examples
# -------------------------------------------------

def show_accepted_examples(n=5):
    print_divider(f"RANDOM ACCEPTED EXAMPLES ({n})")
    chosen = random.sample(sample_examples, min(n, len(sample_examples)))
    for i, ex in enumerate(chosen, 1):
        print(f"\n--- Accepted Example {i} ---")
        print(f"Instruction:\n{ex.get('instruction', '')}\n")
        print(f"Input:\n{ex.get('input', '')}\n")
        print(f"Output:\n{ex.get('output', '')[:3000]}\n")
        if "meta" in ex:
            print("Meta:")
            for k, v in ex["meta"].items():
                print(f"  {k}: {v}")

show_accepted_examples(5)

# -------------------------------------------------
# Random rejected examples
# -------------------------------------------------

def show_rejected_examples(n=5):
    print_divider(f"RANDOM REJECTED EXAMPLES ({n})")
    if len(rejected_rows) == 0:
        print("No rejected rows found.")
        return

    sampled = rejected_rows.sample(min(n, len(rejected_rows)), random_state=42)
    for i, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f"\n--- Rejected Example {i} ---")
        print(f"Reason: {row.get('reject_reason', 'N/A')}")
        prompt = str(row.get("prompt", ""))[:1000]
        text = str(row.get("text", ""))[:1500]
        print(f"Prompt:\n{prompt}\n")
        print(f"Text:\n{text}\n")

show_rejected_examples(5)

# -------------------------------------------------
# Suspicious accepted rows
# -------------------------------------------------

print_divider("SUSPICIOUS ACCEPTED ROWS")

conditions = []
if "word_count" in accepted_meta.columns:
    conditions.append((accepted_meta["word_count"] < 200) | (accepted_meta["word_count"] > 500))
if "character_count" in accepted_meta.columns:
    conditions.append((accepted_meta["character_count"] < 2) | (accepted_meta["character_count"] > 3))
if "estimated_scene_count" in accepted_meta.columns:
    conditions.append((accepted_meta["estimated_scene_count"] < 5) | (accepted_meta["estimated_scene_count"] > 8))

if conditions:
    suspicious_mask = conditions[0]
    for cond in conditions[1:]:
        suspicious_mask = suspicious_mask | cond
    suspicious = accepted_meta[suspicious_mask]
else:
    suspicious = pd.DataFrame()

print(f"Suspicious accepted rows count: {len(suspicious)}")
print(suspicious.head(20))

suspicious.to_csv(OUTPUT_DIR / "suspicious_accepted_rows.csv", index=False)
print("\nSaved suspicious rows to suspicious_accepted_rows.csv")


BASIC COUNTS
Sample examples: 25
Accepted metadata rows: 13983
Rejected rows: 51511

ACCEPTED METADATA SUMMARY
                         count unique  \
source_prompt            13983  13975   
word_count             13983.0    NaN   
sentence_count         13983.0    NaN   
character_count        13983.0    NaN   
estimated_scene_count  13983.0    NaN   
text_token_length      13983.0    NaN   
theme                    13983      9   
title                    13983   8898   
characters               13983  12347   
moral                    13983  12388   

                                                                     top  \
source_prompt          Write an educational story (3-5 paragraphs) ta...   
word_count                                                           NaN   
sentence_count                                                       NaN   
character_count                                                      NaN   
estimated_scene_count                                   

In [8]:
import shutil
from google.colab import files
import os

# Define the name of the folder you want to download (source)
folder_to_download = "/content/processed_children_story_data_v2"

# Define the name of the output zip file (destination)
output_zip_name = "processed_children_story_data_v2"

# Create the zip archive
# 'zip' specifies the archive format, folder_to_download is the directory to archive
shutil.make_archive(output_zip_name, 'zip', folder_to_download)


'/content/processed_children_story_data_v2.zip'